In [1]:
# Repo setup / imports

import os
import pickle
import zipfile
import warnings
import subprocess
from pathlib import Path
from datetime import datetime 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    roc_auc_score
)
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

print(f"numpy={np.__version__}  pandas={pd.__version__}  lightgbm={lgb.__version__}")

REPO_URL = "https://github.com/tongyuguo/HelpHerInvest.git"
REPO_DIR = "HelpHerInvest"

def clone_or_pull():
    """
    Clone the project repository if it does not exist locally.
    Otherwise, pull the latest changes.
    """
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL], check=False)

clone_or_pull()

DATA_DIR = os.path.join(REPO_DIR, "Data")
ARTIFACT_DIR = os.path.join(REPO_DIR, "artifacts")
SPLIT_DIR = os.path.join(ARTIFACT_DIR, "data_splits")
MODEL_DIR = os.path.join(ARTIFACT_DIR, "models")
RESULTS_DIR = os.path.join(ARTIFACT_DIR, "results")

os.makedirs(ARTIFACT_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

STOCKS_CSV = os.path.join(DATA_DIR, "nlp_clean_stock_symbols.csv")
ZIP_PATH = os.path.join(DATA_DIR, "final_dataset_20260224v2.csv.zip")
CSV_IN_ZIP = "final_dataset_20260224v2.csv"

print("Repo directory:", REPO_DIR)
print("Data directory:", DATA_DIR)
print("Artifacts directory:", ARTIFACT_DIR)

numpy=2.2.6  pandas=2.3.3  lightgbm=4.6.0
Already up to date.
Repo directory: HelpHerInvest
Data directory: HelpHerInvest/Data
Artifacts directory: HelpHerInvest/artifacts


In [2]:
# load_stocks

def load_stocks(path=STOCKS_CSV):
    df = pd.read_csv(path)

    required_cols = ["symbol", "document_clean_tfidf", "document_raw"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in stock NLP file: {missing}")

    # TF-IDF text
    df["kw_text"] = df["document_clean_tfidf"].fillna(df["document_raw"]).fillna("")

    # Embedding text
    df["sem_text"] = df["document_raw"].fillna(df["document_clean_tfidf"]).fillna("")

    # Keep rows with at least one usable text field
    df = df[
        (df["kw_text"].astype(str).str.len() > 0) |
        (df["sem_text"].astype(str).str.len() > 0)
    ].copy()

    # Normalize ticker format
    df["symbol"] = df["symbol"].astype(str).str.upper().str.strip()
    df["kw_text"] = df["kw_text"].astype(str)
    df["sem_text"] = df["sem_text"].astype(str)

    return df.reset_index(drop=True)

df_stocks = load_stocks(STOCKS_CSV)

print("Loaded NLP stock rows:", df_stocks.shape)
print(df_stocks[["symbol", "kw_text", "sem_text"]].head(3))

Loaded NLP stock rows: (10283, 8)
  symbol                                            kw_text  \
0   NVDA  nvidia corp technology semiconductors nvidia c...   
1  GOOGL  alphabet communication services internet conte...   
2   AAPL  apple technology consumer electronics apple de...   

                                            sem_text  
0  NVIDIA CORP Technology Semiconductors NVIDIA C...  
1  Alphabet Inc. Communication Services Internet ...  
2  Apple Inc. Technology Consumer Electronics App...  


In [3]:
# load final zipped dataset

with zipfile.ZipFile(ZIP_PATH) as z:
    df = pd.read_csv(z.open(CSV_IN_ZIP))

if "Ticker" not in df.columns or "Date" not in df.columns or "fwd_excess" not in df.columns:
    raise ValueError("Expected columns Ticker, Date, and fwd_excess are required.")

df["Ticker"] = df["Ticker"].astype(str).str.upper().str.strip()
df["Date"] = pd.to_datetime(df["Date"])

print("Main dataset shape:", df.shape)
print("Unique tickers:", df["Ticker"].nunique())
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("Columns:")
print(df.columns.tolist())

Main dataset shape: (302024, 18)
Unique tickers: 1993
Date range: 2010-02-28 00:00:00 to 2026-02-28 00:00:00
Columns:
['Date', 'Ticker', 'mom_1m', 'mom_3m', 'mom_6m', 'mom_12m', 'mom_12m_ex_1m', 'rel_3m_spy', 'rel_6m_spy', 'rel_12m_spy', 'vol_3m', 'vol_6m', 'drawdown_6m', 'drawdown_12m', 'pct_above_200dma', 'adj_close', 'fwd_excess', 'fwd_return']


In [4]:
# create unique ticker dataframe 

def create_unique_ticker_df(df):
    if "Ticker" not in df.columns:
        raise ValueError("Column 'Ticker' not found in dataframe.")
    
    unique_ticker = (
        df["Ticker"]
        .dropna()
        .astype(str)
        .str.upper()
        .str.strip()
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
        .to_frame(name="Ticker")
    )
    
    return unique_ticker

unique_ticker = create_unique_ticker_df(df)

print(unique_ticker.head())
print("Number of unique tickers:", len(unique_ticker))

  Ticker
0      A
1     AA
2    AAL
3   AAON
4    AAP
Number of unique tickers: 1993


In [5]:
# create y

# Binary target:
# 1 if future excess return is positive, 0 otherwise
df = df.copy()
df["y"] = (df["fwd_excess"] > 0).astype(int)

print("Positive fwd_excess count:", (df["fwd_excess"] > 0).sum())
print("Negative fwd_excess count:", (df["fwd_excess"] < 0).sum())
print("Zero fwd_excess count:", (df["fwd_excess"] == 0).sum())

print("\nTarget distribution:")
print(df["y"].value_counts(dropna=False).sort_index())
print(df["y"].value_counts(normalize=True).sort_index())

Positive fwd_excess count: 143948
Negative fwd_excess count: 151730
Zero fwd_excess count: 287

Target distribution:
y
0    158076
1    143948
Name: count, dtype: int64
y
0    0.523389
1    0.476611
Name: proportion, dtype: float64


In [6]:
# create df_rank

# Ranked feature columns from the current notebook structure
rank_cols = [
    "mom_1m", "mom_3m", "mom_6m", "mom_12m", "mom_12m_ex_1m",
    "rel_3m_spy", "rel_6m_spy", "rel_12m_spy",
    "vol_3m", "vol_6m",
    "drawdown_6m", "drawdown_12m",
    "pct_above_200dma"
]

required_for_model = ["Date", "Ticker", "fwd_excess", "y", "adj_close"] + rank_cols
missing_required = [c for c in required_for_model if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns for ranking/modeling: {missing_required}")

# Drop rows only if required model columns are missing
df_model_base = df.dropna(subset=required_for_model).copy()

df_rank = df_model_base.copy()

# Cross-sectional percentile rank by Date
for col in rank_cols:
    df_rank[col] = df_rank.groupby("Date")[col].rank(pct=True)

# Rank the forward excess return by Date
df_rank["fwd_rank"] = df_rank.groupby("Date")["fwd_excess"].rank(pct=True)

# 5-bin label
df_rank["target"] = pd.cut(
    df_rank["fwd_rank"],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=[0, 1, 2, 3, 4],
    include_lowest=True
)

print("df_rank shape:", df_rank.shape)
print(df_rank.head(3))

df_rank shape: (274511, 21)
            Date Ticker    mom_1m    mom_3m    mom_6m   mom_12m  \
13389 2011-01-31   NVDA  0.997496  0.995826  0.995826  0.814691   
13390 2011-01-31  GOOGL  0.480801  0.142738  0.636060  0.281302   
13391 2011-01-31   AAPL  0.729549  0.623539  0.766277  0.909015   

       mom_12m_ex_1m  rel_3m_spy  rel_6m_spy  rel_12m_spy  ...    vol_6m  \
13389       0.137730    0.995826    0.995826     0.814691  ...  0.928214   
13390       0.276294    0.142738    0.636060     0.281302  ...  0.438230   
13391       0.889816    0.623539    0.766277     0.909015  ...  0.222871   

       drawdown_6m  drawdown_12m  pct_above_200dma  adj_close  fwd_excess  \
13389     0.744157      0.790067          0.995826     0.5483     -0.2287   
13390     0.377295      0.476628          0.515025    14.9114     -0.1585   
13391     0.744157      0.790067          0.747913    10.1670     -0.0330   

       fwd_return  y  fwd_rank  target  
13389     -0.1639  0  0.030050       0  
13390  

In [7]:
print(os.getcwd())

/home/jupyter-guotq


In [8]:
from pprint import pprint

#API 
from dotenv import load_dotenv
import os
load_dotenv() 

True

In [11]:
#testing if the API key is working

from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-5.4",
    input="tell me about boston college, in 100 words or less"
)

print(response.output_text)

Boston College is a private Jesuit research university in Chestnut Hill, Massachusetts, founded in 1863. It’s known for strong academics, especially in business, education, law, and the liberal arts, plus a focus on ethics, service, and holistic education. BC has a suburban campus near Boston, a vibrant student life, and competitive NCAA Division I athletics, especially football and hockey. Though called “Boston” College, its main campus is not in downtown Boston. It’s considered selective and combines rigorous academics with Jesuit Catholic traditions and a strong alumni network.


In [ ]:
from openai import OpenAI
import pandas as pd
import io
import json

client = OpenAI()

def get_30_unique_tickers(unique_ticker: pd.DataFrame, user_input: str, model: str = "gpt-5-mini") -> pd.DataFrame:

    if "Ticker" not in unique_ticker.columns:
        raise ValueError("unique_ticker must contain a 'Ticker' column.")

    # Clean ticker column
    valid_tickers = (
        unique_ticker["Ticker"]
        .dropna()
        .astype(str)
        .str.upper()
        .str.strip()
        .drop_duplicates()
    )

    valid_ticker_set = set(valid_tickers)

    # Convert dataframe to in-memory CSV
    ticker_buffer = io.BytesIO()
    valid_tickers.to_frame(name="Ticker").to_csv(ticker_buffer, index=False)
    ticker_buffer.seek(0)
    ticker_buffer.name = "unique_ticker.csv"

    # Upload in-memory file
    uploaded_file = client.files.create(file=ticker_buffer, purpose="user_data")

    # Call model
    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "developer",
                "content": [
                    {
                        "type": "input_text",
                        "text": (
                            "You are selecting stock tickers for a downstream model.\n"
                            "You must use the attached CSV file as the only allowed universe of tickers.\n"
                            "Based on the user's interest, return exactly 30 unique ticker symbols.\n"
                            "Rules:\n"
                            "- Only return tickers that exist in the attached file.\n"
                            "- Return exactly 30 unique tickers.\n"
                            "- Do not include explanations.\n"
                            "- Do not include markdown.\n"
                            '- Return valid JSON in this exact format: {"tickers": ["AAPL", "MSFT", "..."]}'
                        )
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {"type": "input_file", "file_id": uploaded_file.id},
                    {"type": "input_text", "text": user_input}
                ]
            }
        ]
    )

    # Parse JSON output
    try:
        result = json.loads(response.output_text)
        tickers = result["tickers"]
    except Exception as e:
        raise ValueError(f"Model output was not valid JSON: {response.output_text}") from e

    # Clean + validate output
    cleaned = []
    seen = set()

    for t in tickers:
        t = str(t).upper().strip()
        if t in valid_ticker_set and t not in seen:
            cleaned.append(t)
            seen.add(t)

    if len(cleaned) != 30:
        raise ValueError(
            f"Expected 30 valid unique tickers, but got {len(cleaned)} after validation.\n"
            f"Raw model output: {response.output_text}"
        )

    # Return dataframe
    selected_tickers_df = pd.DataFrame({"Ticker": cleaned})
    return selected_tickers_df

In [13]:
user_input = "I like the fashion industry."

selected_tickers_df = get_30_unique_tickers(unique_ticker, user_input)

print(selected_tickers_df)
print(selected_tickers_df.shape)

   Ticker
0     ANF
1     AEO
2     GAP
3    CPRI
4    CROX
5     BKE
6    BURL
7    BOOT
8      EL
9    COTY
10    ELF
11   ETSY
12   AMZN
13   EBAY
14   BABA
15     JD
16    GIL
17   COST
18   DLTR
19   DECK
20   COLM
21    BBY
22     BJ
23    CVS
24   AAPL
25  GOOGL
26   BBWI
27    APP
28   APPF
29   APPN
(30, 1)
